# Decision model (noul + choice): Qwen3-1.7B + LoRA

Варіанти відповіді позначаються літерами `A..H`; модель робить один forward pass, ймовірності варіантів — softmax логітів літер із температурою $T$.

## Конфігурація

In [11]:
import json, random, re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

CFG = dict(
    model_name      = "Qwen/Qwen3-1.7B",
    model_tag       = "s1-qwen3-1.7b-lora",
    save_dir        = "decision_lora",
    seed            = 3407,
    max_seq_len     = 1280,
    max_state_train = 320,
    max_state_eval  = 768,
    letters         = list("ABCDEFGH"),
    uk_share        = 0.4,
    n_train         = {"boolq":     {"noul": 2500},
                       "banking77": {"choice": 2000, "noul": 1500},
                       "ag_news":   {"choice": 1500, "noul": 1000},
                       "emotion":   {"choice": 1500, "noul": 1000},
                       "mnli":      {"choice": 1500, "noul": 1500},
                       "imdb":      {"noul": 1000}},
    n_eval          = 80,
    n_ood           = {"clinc150": {"choice": 400, "noul": 400}, "sms_spam": {"noul": 400},
                       "typed_decisions": {"rows": 150}},
    eval_batch_size = 16,
    lora            = dict(r=16, lora_alpha=16, lora_dropout=0.0,
                           target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                           "gate_proj", "up_proj", "down_proj"]),
    train           = dict(per_device_train_batch_size=8, gradient_accumulation_steps=2,
                           num_train_epochs=1, learning_rate=2e-4, warmup_steps=30,
                           lr_scheduler_type="cosine", weight_decay=0.01, logging_steps=20),
)

SEED, LETTERS = CFG["seed"], CFG["letters"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## Модель + LoRA

In [12]:
BF16  = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(CFG["model_name"], dtype=DTYPE, device_map={"": 0})
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", bias="none", **CFG["lora"]))
DEVICE = model.device
model.print_trainable_parameters()

Loading weights: 100%|██████████| 311/311 [00:05<00:00, 53.36it/s]


trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


## Промпт

In [13]:
SYSTEM_PROMPT = ("You are a decision model. Read the text and answer the question "
                 "by choosing exactly one option. Reply with the option letter only.")

LETTER_IDS = []
for L in LETTERS:
    ids = tokenizer.encode(L, add_special_tokens=False)
    assert len(ids) == 1, f"{L} -> {ids}"
    LETTER_IDS.append(ids[0])
LETTER_IDS = torch.tensor(LETTER_IDS)

def truncate_text(text, max_tokens):
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= max_tokens else tokenizer.decode(ids[:max_tokens]) + " ..."

def build_prompt_ids(state, question, options, max_state_tokens):
    opts = "\n".join(f"{LETTERS[i]}) {o}" for i, o in enumerate(options))
    user = (f"Text:\n{truncate_text(state, max_state_tokens)}\n\nQuestion: {question}\n\n"
            f"Options:\n{opts}\n\nAnswer with one letter.")
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)
    return tokenizer(text, add_special_tokens=False)["input_ids"]

## Дані

In [14]:
YES_NO = {"en": ["Yes", "No"], "uk": ["Так", "Ні"]}

CLASSIFICATION = {
    "banking77": dict(
        path="PolyAI/banking77", splits=("train", "test"), keys=None, descs=None,
        choice_q={"en": ["What is the customer's request about?", "Which intent best describes this message?",
                         "Route this banking request to the right topic."],
                  "uk": ["Про що звертається клієнт?", "Який намір найкраще описує це повідомлення?",
                         "До якої теми належить цей запит клієнта банку?"]},
        noul_q={"en": ["Is the customer asking about: {x}?", 'Is this request related to "{x}"?'],
                "uk": ["Чи питає клієнт про: {x}?", "Чи стосується цей запит теми «{x}»?"]}),
    "ag_news": dict(
        path="fancyzhx/ag_news", splits=("train", "test"), keys=["world", "sports", "business", "sci_tech"],
        descs={"en": ["world news, politics, international affairs", "sports", "business, economy, finance",
                      "science and technology"],
               "uk": ["світові новини, політика, міжнародні події", "спорт", "бізнес, економіка, фінанси",
                      "наука і технології"]},
        choice_q={"en": ["What is the topic of this news article?", "Which newspaper section does this article belong to?"],
                  "uk": ["Яка тема цієї новини?", "До якої рубрики газети належить ця стаття?"]},
        noul_q={"en": ["Is this article about {x}?", "Does this news belong to the category: {x}?"],
                "uk": ["Чи ця стаття про {x}?", "Чи належить ця новина до категорії: {x}?"]}),
    "emotion": dict(
        path="dair-ai/emotion", splits=("train", "test"),
        keys=["sadness", "joy", "love", "anger", "fear", "surprise"],
        descs={"en": ["sadness, disappointment, grief", "joy, happiness, satisfaction", "love, affection, tenderness",
                      "anger, irritation, annoyance", "fear, anxiety, worry", "surprise, astonishment"],
               "uk": ["сум, розчарування, горе", "радість, щастя, задоволення", "любов, ніжність, прихильність",
                      "гнів, роздратування", "страх, тривога, хвилювання", "здивування, подив"]},
        choice_q={"en": ["What emotion does the author express?", "Which feeling dominates in this message?"],
                  "uk": ["Яку емоцію виражає автор?", "Яке почуття переважає в цьому повідомленні?"]},
        noul_q={"en": ["Does the author express {x}?", "Is the main emotion of this text: {x}?"],
                "uk": ["Чи виражає автор {x}?", "Чи головна емоція цього тексту — {x}?"]}),
}

BOOLQ = dict(path="google/boolq", splits=("train", "validation"))

MNLI = dict(
    path="nyu-mll/multi_nli", splits=("train", "validation_matched"),
    keys=["entailment", "neutral", "contradiction"],
    descs={"en": ["the statement follows from the text", "the text neither confirms nor refutes the statement",
                  "the statement contradicts the text"],
           "uk": ["твердження випливає з тексту", "текст не підтверджує і не спростовує твердження",
                  "твердження суперечить тексту"]},
    choice_q={"en": ['How does the statement "{h}" relate to the text?'],
              "uk": ["Як твердження «{h}» співвідноситься з текстом?"]},
    noul_q={"en": ['Does it follow from the text that "{h}"?', 'Is the statement "{h}" supported by the text?'],
            "uk": ["Чи випливає з тексту, що «{h}»?", "Чи підтверджує текст твердження «{h}»?"]})

IMDB = dict(
    path="stanfordnlp/imdb", splits=("train", "test"),
    noul_q={"en": [("Is this movie review positive?", 1), ("Is the reviewer unhappy with the movie?", 0),
                   ("Did the author like the film?", 1)],
            "uk": [("Чи позитивний цей відгук на фільм?", 1), ("Чи автор незадоволений фільмом?", 0),
                   ("Чи сподобався автору фільм?", 1)]})

CLINC = dict(
    path="clinc/clinc_oos", config="plus", split="test", descs=None,
    choice_q={"en": ["What does the user want?", "Which intent matches this request?"],
              "uk": ["Чого хоче користувач?", "Який намір відповідає цьому запиту?"]},
    noul_q={"en": ["Does the user want: {x}?"], "uk": ["Чи хоче користувач: {x}?"]})

SMS = dict(
    path="ucirvine/sms_spam", split="train",
    noul_q={"en": ["Is this message spam?", "Is this an unsolicited advertising or scam message?"],
            "uk": ["Чи це повідомлення — спам?", "Чи це небажана реклама або шахрайське повідомлення?"]})

TYPED = dict(path="LocalLLaMA/typed-decisions", config="all", split="test")

In [15]:
rng = random.Random(SEED)

def pick_lang():
    return "uk" if rng.random() < CFG["uk_share"] else "en"

def make_noul(state, question, answer, lang, source):
    options, gold = list(YES_NO[lang]), 0 if answer else 1
    if rng.random() < 0.5:
        options.reverse(); gold = 1 - gold
    return dict(state=state, question=question, options=options, gold=gold, qtype="noul", source=source)

def make_choice(state, question, keys, descs, gold, source):
    k = rng.randint(2, min(len(LETTERS), len(keys)))
    idx = rng.sample([i for i in range(len(keys)) if i != gold], k - 1) + [gold]
    rng.shuffle(idx)
    fmt = {"both": lambda i: f"{keys[i]}: {descs[i]}",
           "key":  lambda i: keys[i],
           "desc": lambda i: descs[i]}[rng.choice(["both", "both", "key", "desc"])]
    return dict(state=state, question=question, options=[fmt(i) for i in idx], gold=idx.index(gold),
                qtype="choice", source=source)

def sample_rows(ds, n):
    ds = ds.shuffle(seed=SEED)
    return ds.select(range(min(n, len(ds)))).to_list()

def balanced_rows(ds, is_pos, n):
    ds = ds.shuffle(seed=SEED).select(range(min(len(ds), 20 * n)))
    out = sample_rows(ds.filter(is_pos), n // 2) + sample_rows(ds.filter(lambda r: not is_pos(r)), n - n // 2)
    rng.shuffle(out)
    return out

In [16]:
def classification_examples(name, spec, ds, n):
    keys = spec["keys"] or ds.features["label"].names
    rows = sample_rows(ds, n.get("choice", 0) + n.get("noul", 0))
    out = []
    for i, r in enumerate(rows):
        lang  = pick_lang()
        descs = spec["descs"][lang] if spec["descs"] else [k.replace("_", " ") for k in keys]
        y = r["label"]
        if i < n.get("choice", 0):
            out.append(make_choice(r["text"], rng.choice(spec["choice_q"][lang]), keys, descs, y, name))
        else:
            ask_true = rng.random() < 0.5
            c = y if ask_true else rng.choice([j for j in range(len(keys)) if j != y])
            out.append(make_noul(r["text"], rng.choice(spec["noul_q"][lang]).format(x=descs[c]),
                                 ask_true, lang, name))
    return out

def boolq_examples(ds, n):
    out = []
    for r in balanced_rows(ds, lambda r: r["answer"], n["noul"]):
        q = r["question"].strip()
        out.append(make_noul(r["passage"], q[0].upper() + q[1:] + "?", bool(r["answer"]), pick_lang(), "boolq"))
    return out

def mnli_examples(ds, n):
    ds, out = ds.filter(lambda b: [l in (0, 1, 2) for l in b["label"]], batched=True), []
    for r in sample_rows(ds, n["choice"]):
        lang = pick_lang()
        q = rng.choice(MNLI["choice_q"][lang]).format(h=r["hypothesis"])
        out.append(make_choice(r["premise"], q, MNLI["keys"], MNLI["descs"][lang], r["label"], "mnli"))
    for r in balanced_rows(ds, lambda r: r["label"] == 0, n["noul"]):
        lang = pick_lang()
        q = rng.choice(MNLI["noul_q"][lang]).format(h=r["hypothesis"])
        out.append(make_noul(r["premise"], q, r["label"] == 0, lang, "mnli"))
    return out

def imdb_examples(ds, n):
    out = []
    for r in sample_rows(ds, n["noul"]):
        lang = pick_lang()
        q, yes_label = rng.choice(IMDB["noul_q"][lang])
        out.append(make_noul(r["text"].replace("<br />", " "), q, r["label"] == yes_label, lang, "imdb"))
    return out

def sms_examples(ds, n):
    out = []
    for r in balanced_rows(ds, lambda r: r["label"] == 1, n["noul"]):
        lang = pick_lang()
        out.append(make_noul(r["sms"], rng.choice(SMS["noul_q"][lang]), r["label"] == 1, lang, "sms_spam"))
    return out

def parse_bool(g):
    if isinstance(g, dict):
        g = next((g[k] for k in ("label", "answer", "value", "noul", "probability_true") if k in g), None)
    if isinstance(g, bool):         return g
    if isinstance(g, (int, float)): return g > 0.5
    if isinstance(g, str):          return {"true": True, "yes": True, "1": True,
                                            "false": False, "no": False, "0": False}.get(g.strip().lower())
    return None

def typed_decisions_examples(ds, n):
    out = []
    for r in sample_rows(ds, n["rows"]):
        try:
            qs, gold = json.loads(r["questions"]), json.loads(r["gold"])
        except (TypeError, ValueError):
            continue
        state = r["state"] if isinstance(r["state"], str) else json.dumps(r["state"], ensure_ascii=False)
        for name, q in qs.items():
            g = gold.get(name)
            if g is None:
                continue
            if q.get("type") == "choice":
                crit = q["criteria"]
                keys = list(crit)
                options = [f"{k}: {v}" for k, v in crit.items()] if isinstance(crit, dict) else keys
                lab = g.get("label") if isinstance(g, dict) else g
                if lab in keys and len(keys) <= len(LETTERS):
                    out.append(dict(state=state, question=q["instructions"], options=options,
                                    gold=keys.index(lab), qtype="choice", source="typed_decisions"))
            elif q.get("type") == "noul" and parse_bool(g) is not None:
                out.append(dict(state=state, question=q["instructions"], options=list(YES_NO["en"]),
                                gold=0 if parse_bool(g) else 1, qtype="noul", source="typed_decisions"))
    return out

def build_ood():
    clinc = load_dataset(CLINC["path"], CLINC["config"], split=CLINC["split"])
    names = clinc.features["intent"].names
    keep  = [i for i, name in enumerate(names) if name != "oos"]
    remap = {old: new for new, old in enumerate(keep)}
    clinc = clinc.filter(lambda r: r["intent"] in remap).map(lambda r: {"label": remap[r["intent"]]})
    out  = classification_examples("clinc150", {**CLINC, "keys": [names[i] for i in keep]}, clinc,
                                   CFG["n_ood"]["clinc150"])
    out += sms_examples(load_dataset(SMS["path"], split=SMS["split"]), CFG["n_ood"]["sms_spam"])
    out += typed_decisions_examples(load_dataset(TYPED["path"], TYPED["config"], split=TYPED["split"]),
                                    CFG["n_ood"]["typed_decisions"])
    return out

In [17]:
DATA = {name: load_dataset(spec["path"])
        for name, spec in [*CLASSIFICATION.items(), ("boolq", BOOLQ), ("mnli", MNLI), ("imdb", IMDB)]}

def build_split(train):
    k = 0 if train else 1
    n = (lambda src: CFG["n_train"][src]) if train else \
        (lambda src: {q: 2 * CFG["n_eval"] for q in CFG["n_train"][src]})
    out = []
    for name, spec in CLASSIFICATION.items():
        out += classification_examples(name, spec, DATA[name][spec["splits"][k]], n(name))
    out += boolq_examples(DATA["boolq"][BOOLQ["splits"][k]], n("boolq"))
    out += mnli_examples(DATA["mnli"][MNLI["splits"][k]], n("mnli"))
    out += imdb_examples(DATA["imdb"][IMDB["splits"][k]], n("imdb"))
    rng.shuffle(out)
    return out

def encode(examples, max_state):
    out = []
    for ex in examples:
        ids = build_prompt_ids(ex["state"], ex["question"], ex["options"], max_state)
        if len(ids) <= CFG["max_seq_len"]:
            out.append({**ex, "input_ids": ids, "n_opt": len(ex["options"]), "length": len(ids)})
    return out

train_ex = encode(build_split(train=True), CFG["max_state_train"])
eval_ex  = encode(build_split(train=False), CFG["max_state_eval"])
val_ex, test_ex = eval_ex[0::2], eval_ex[1::2]
ood_ex   = encode(build_ood(), CFG["max_state_eval"])
train_ds = Dataset.from_list([{k: e[k] for k in ("input_ids", "gold", "n_opt", "length")} for e in train_ex])
print(f"train {len(train_ex)} | val {len(val_ex)} | test {len(test_ex)} | ood {len(ood_ex)}")

train 15000 | val 800 | test 800 | ood 1650


## Логіти літер

In [18]:
def letter_scores(model, input_ids, attention_mask):
    base   = model.get_base_model()
    hidden = base.model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
    last   = attention_mask.sum(dim=1) - 1
    h      = hidden[torch.arange(hidden.size(0), device=hidden.device), last]
    W      = base.lm_head.weight[LETTER_IDS.to(base.lm_head.weight.device)]
    return h.float() @ W.float().T

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids  = torch.full((len(batch), L), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        ids[i, :len(b["input_ids"])]  = torch.tensor(b["input_ids"])
        mask[i, :len(b["input_ids"])] = 1
    return {"input_ids": ids, "attention_mask": mask,
            "gold":  torch.tensor([b.get("gold", 0) for b in batch]),
            "n_opt": torch.tensor([b["n_opt"] for b in batch])}

@torch.no_grad()
def score_examples(examples, batch_size=CFG["eval_batch_size"]):
    model.eval()
    order, out = np.argsort([len(e["input_ids"]) for e in examples]), [None] * len(examples)
    for s in range(0, len(order), batch_size):
        idx   = order[s:s + batch_size]
        batch = collate([examples[i] for i in idx])
        with torch.autocast(device_type=DEVICE.type, dtype=DTYPE):
            sc = letter_scores(model, batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
        sc = sc.float().cpu().numpy()
        for j, i in enumerate(idx):
            out[i] = sc[j, :examples[i]["n_opt"]]
    return out

## Baseline (zero-shot)

In [19]:
with model.disable_adapter():
    base_logits = {"test": score_examples(test_ex), "ood": score_examples(ood_ex)}

## Навчання

In [20]:
class LetterTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        scores = letter_scores(model, inputs["input_ids"], inputs["attention_mask"])
        n_opt  = inputs["n_opt"]
        scores = scores.masked_fill(torch.arange(scores.size(1), device=scores.device)[None, :] >= n_opt[:, None],
                                    float("-inf"))
        loss = F.cross_entropy(scores, inputs["gold"])
        return (loss, scores) if return_outputs else loss

args = TrainingArguments(
    output_dir="outputs", optim="adamw_torch", bf16=BF16, fp16=not BF16,
    save_strategy="no", report_to="none", train_sampling_strategy="group_by_length",
    remove_unused_columns=False, label_names=["gold"], seed=SEED, **CFG["train"])
trainer = LetterTrainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)
trainer.train()

Step,Training Loss
20,3.546691
40,0.614910
60,0.545569
80,0.486370
100,0.369707
120,0.477415
140,0.420289
160,0.395499
180,0.364041
200,0.281494


TrainOutput(global_step=938, training_loss=0.41929801389860955, metrics={'train_runtime': 1613.5846, 'train_samples_per_second': 9.296, 'train_steps_per_second': 0.581, 'total_flos': 1.8876922450182144e+16, 'train_loss': 0.41929801389860955, 'epoch': 1.0})

## Калібрування та оцінка

In [21]:
def softmax(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    e = np.exp(z - z.max())
    return e / e.sum()

def ece_score(conf, correct, n_bins=15):
    conf, correct = np.asarray(conf), np.asarray(correct, dtype=float)
    edges, ece = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def predictions(examples, logits, T=1.0):
    rows = []
    for e, z in zip(examples, logits):
        p = softmax(z, T)
        rows.append(dict(source=e["source"], qtype=e["qtype"], conf=p.max(),
                         correct=int(p.argmax() == e["gold"]), nll=-np.log(p[e["gold"]] + 1e-12)))
    return pd.DataFrame(rows)

def metrics(logits, T=1.0):
    rows = []
    for set_name, examples in [("test", test_ex), ("ood", ood_ex)]:
        df = predictions(examples, logits[set_name], T)
        groups = list(df.groupby(["source", "qtype"])) + [(("ALL", q), g) for q, g in df.groupby("qtype")]
        rows += [dict(set=set_name, source=s, qtype=q, n=len(g), acc=g.correct.mean(),
                      ece=ece_score(g.conf, g.correct), nll=g.nll.mean()) for (s, q), g in groups]
    return pd.DataFrame(rows).set_index(["set", "source", "qtype"])

val_logits = score_examples(val_ex)
ft_logits  = {"test": score_examples(test_ex), "ood": score_examples(ood_ex)}
T_grid = np.exp(np.linspace(np.log(0.1), np.log(10.0), 161))
T_STAR = float(T_grid[np.argmin([predictions(val_ex, val_logits, T).nll.mean() for T in T_grid])])
print(f"T* = {T_STAR:.3f}")

mb, m1, mT = metrics(base_logits), metrics(ft_logits), metrics(ft_logits, T_STAR)
results = pd.DataFrame({"n": mb["n"], "acc_baseline": mb["acc"], "acc_ft": m1["acc"],
                        "ece_baseline": mb["ece"], "ece_ft": m1["ece"], "ece_ft_T*": mT["ece"],
                        "nll_baseline": mb["nll"], "nll_ft": m1["nll"], "nll_ft_T*": mT["nll"]})
results.round(3)

T* = 1.029


n  acc_baseline  acc_ft  ece_baseline  ece_ft  \
set  source          qtype                                                      
test ag_news         choice    71         0.803   0.901         0.194   0.034   
                     noul      88         0.830   0.886         0.161   0.061   
     banking77       choice    78         0.897   0.936         0.098   0.050   
                     noul      95         0.874   0.937         0.128   0.062   
     boolq           noul      68         0.824   0.838         0.176   0.078   
     emotion         choice    80         0.575   0.838         0.417   0.109   
                     noul      78         0.641   0.910         0.362   0.069   
     imdb            noul      86         0.872   0.977         0.120   0.045   
     mnli            choice    84         0.726   0.893         0.281   0.062   
                     noul      72         0.792   0.931         0.217   0.070   
     ALL             choice   313         0.748   0.891         0.249   0.037   
                     noul     487         0.809   0.916         0.188   0.015   
ood  clinc150        choice   400         0.952   0.982         0.045   0.010   
                     noul     400         0.878   0.932         0.116   0.047   
     sms_spam        noul     400         0.855   0.682         0.137   0.112   
     typed_decisions choice   232         0.470   0.539         0.517   0.099   
                     noul     218         0.661   0.679         0.338   0.130   
     ALL             choice   632         0.775   0.820         0.218   0.035   
                     noul    1018         0.822   0.780         0.170   0.075   

                             ece_ft_T*  nll_baseline  nll_ft  nll_ft_T*  
set  source          qtype                                               
test ag_news         choice      0.031         4.224   0.271      0.270  
                     noul        0.058         2.280   0.288      0.286  
     banking77       choice      0.049         1.180   0.149      0.149  
                     noul        0.065         2.236   0.136      0.137  
     boolq           noul        0.072         3.711   0.363      0.361  
     emotion         choice      0.100         7.773   0.447      0.446  
                     noul        0.072         7.428   0.299      0.300  
     imdb            noul        0.048         1.719   0.095      0.098  
     mnli            choice      0.083         3.946   0.216      0.217  
                     noul        0.065         4.347   0.185      0.186  
     ALL             choice      0.035         4.298   0.271      0.270  
                     noul        0.016         3.502   0.221      0.222  
ood  clinc150        choice      0.012         0.613   0.075      0.076  
                     noul        0.054         1.478   0.170      0.171  
     sms_spam        noul        0.109         1.623   0.562      0.559  
     typed_decisions choice      0.098         9.069   1.106      1.101  
                     noul        0.125         6.415   0.676      0.671  
     ALL             choice      0.037         3.717   0.453      0.452  
                     noul        0.072         2.592   0.433      0.431

## API: `systemone(request)`

In [22]:
CYRILLIC = re.compile(r"[а-яіїєґ]", re.IGNORECASE)

def option_probs(state, question, options, T):
    ids = build_prompt_ids(state, question, options, CFG["max_state_eval"])
    return softmax(score_examples([{"input_ids": ids, "n_opt": len(options)}])[0], T), len(ids)

def systemone(request, T=None):
    T = T_STAR if T is None else T
    answers, input_tokens = {}, 0
    for name, q in request["questions"].items():
        qtype, instr = q.get("type"), q.get("instructions", "")
        if qtype == "noul":
            lang = "uk" if CYRILLIC.search(instr) else "en"
            p, n_tok = option_probs(request["state"], instr, YES_NO[lang], T)
            answers[name] = {"type": "noul", "noul": round(float(p[0]), 4)}
        elif qtype == "choice":
            crit = q["criteria"]
            keys = list(crit)
            if len(keys) > len(LETTERS):
                raise ValueError(f"{name}: max {len(LETTERS)} options")
            options = [f"{k}: {v}" for k, v in crit.items()] if isinstance(crit, dict) else keys
            p, n_tok = option_probs(request["state"], instr, options, T)
            order = np.argsort(-p)
            answers[name] = {"type": "choice", "choice": keys[order[0]],
                             "probabilities": {keys[i]: round(float(p[i]), 4) for i in order},
                             "confidence": round(float(p.max()), 4)}
        else:
            answers[name] = {"type": qtype, "error": f"unsupported question type: {qtype}"}
            continue
        input_tokens += n_tok
    return {"model": CFG["model_tag"], "answers": answers,
            "usage": {"input_tokens": input_tokens, "output_tokens": 0}}

In [23]:
request = {
    "state": "Вітаю! З мене двічі списали оплату за місячну підписку. Поверніть, будь ласка, зайве.",
    "questions": {
        "department":   {"type": "choice", "instructions": "Яка команда має обробити цей запит?",
                         "criteria": {"billing": "Рахунки, оплати, повернення коштів",
                                      "technical": "Баги, збої, помилки системи",
                                      "other": "Усе інше"}},
        "wants_refund": {"type": "noul", "instructions": "Клієнт явно просить повернути кошти?"},
    },
}
print(json.dumps(systemone(request), ensure_ascii=False, indent=2))

{
  "model": "s1-qwen3-1.7b-lora",
  "answers": {
    "department": {
      "type": "choice",
      "choice": "billing",
      "probabilities": {
        "billing": 0.9785,
        "other": 0.0139,
        "technical": 0.0076
      },
      "confidence": 0.9785
    },
    "wants_refund": {
      "type": "noul",
      "noul": 0.9713
    }
  },
  "usage": {
    "input_tokens": 290,
    "output_tokens": 0
  }
}


## Збереження

In [24]:
model.save_pretrained(CFG["save_dir"])
tokenizer.save_pretrained(CFG["save_dir"])
with open(f"{CFG['save_dir']}/decision_config.json", "w", encoding="utf-8") as f:
    json.dump({"base_model": CFG["model_name"], "temperature": T_STAR, "letters": LETTERS,
               "system_prompt": SYSTEM_PROMPT, "max_state_tokens": CFG["max_state_eval"]},
              f, ensure_ascii=False, indent=2)
results.round(4).to_csv(f"{CFG['save_dir']}/results.csv")